# 8.0 - Simulacao local do Job do Databricks

Faz aqui, com pandas, o mesmo que `databricks/pluma_job_5min.py` faz no Databricks com Spark:

1. le as linhas novas da **tabela crua** (as mesmas colunas de `data/raw`), desde 60 min antes do ultimo instante ja salvo;
2. `tabela_previsao`: uma linha por instante (score da camera, probabilidade e alarme de cada cabeca);
3. `tabela_recomendacao`: uma linha por tag a mudar, nos instantes com alarme;
4. grava sem duplicar (no Databricks e um MERGE; aqui, juntar e tirar repetidos).

Nao treina nada: usa os `.pkl` e o `limiares.json` de `models/`.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
from datetime import timedelta

from ml_atmospheric_emission.config import PATH_REPORTS
from ml_atmospheric_emission.dataset import load_raw
from ml_atmospheric_emission.pipeline import tabela_previsao, tabela_recomendacao

In [3]:
#The raw table (in the Databricks it is a Delta table with the same columns)
bruta = load_raw()
coluna_tempo = bruta.columns[0]   #in the Excel the time is the first column, without name
bruta = bruta.rename(columns={coluna_tempo: 'Data'}).sort_values('Data').reset_index(drop=True)
print('Tabela crua:', bruta.shape, '| de', bruta['Data'].min(), 'ate', bruta['Data'].max())
bruta.tail(3)

Tabela crua: (82182, 11) | de 2026-01-01 00:05:00 ate 2026-10-13 08:30:00


,Data,nn_ok_smog,nn_score_smog,int_sd,int_med,306-AIT-004-2,306-AIT-004-5,306FE002-AI001-R,406-AI-004-2,406-AI-004-4,406-AIT-001
82179,2026-10-13 08:20:00,0.0,0.001279,3.0,5.334077,54.422600,137.762843,2.409530,63.983227,189.094398,3.879804
82180,2026-10-13 08:25:00,0.0,0.000978,9.0,11.835665,56.460173,141.337282,1.841905,73.699460,203.883847,2.368845
82181,2026-10-13 08:30:00,0.0,0.000677,8.0,10.586340,58.261623,151.767673,2.979544,77.782647,185.112225,3.404892


In [4]:
def juntar(tabela, novos, chaves):
    #Same as the MERGE of the Databricks: adds only the rows whose keys are not in the table yet
    if len(novos) == 0:
        return tabela
    tabela = pd.concat([tabela, novos]).drop_duplicates(subset=chaves, keep='first')
    return tabela.sort_values(chaves).reset_index(drop=True)


def rodar_job(bruta_ate_agora, tabela_prev, tabela_rec):
    #One run of the job (the same steps of databricks/pluma_job_5min.py)
    ultimo = None
    if len(tabela_prev) > 0:
        ultimo = tabela_prev['Data'].max()
    ds_bruto = bruta_ate_agora
    if ultimo is not None:
        ds_bruto = ds_bruto[ds_bruto['Data'] > ultimo - timedelta(minutes=60)]
    previsao = tabela_previsao(ds_bruto, a_partir_de=ultimo)
    inicio_recomendacao = ultimo
    if inicio_recomendacao is None:
        inicio_recomendacao = ds_bruto['Data'].max() - timedelta(days=1)
    recomendacao = tabela_recomendacao(ds_bruto, a_partir_de=inicio_recomendacao)
    print('linhas cruas lidas:', len(ds_bruto), '| previsoes novas:', len(previsao), '| recomendacoes novas:', len(recomendacao))
    return juntar(tabela_prev, previsao, ['Data']), juntar(tabela_rec, recomendacao, ['Data', 'variavel'])

## Primeira execucao (as tabelas de saida ainda nao existem)

A tabela crua vai ate 1 hora antes do fim: as previsoes saem para todo o historico, as recomendacoes para o ultimo dia.

In [5]:
tabela_prev = pd.DataFrame()
tabela_rec = pd.DataFrame()
agora = bruta['Data'].max() - timedelta(hours=1)
tabela_prev, tabela_rec = rodar_job(bruta[bruta['Data'] <= agora], tabela_prev, tabela_rec)

linhas cruas lidas: 82170 | previsoes novas: 15062 | recomendacoes novas: 759


## Proximas execucoes: a cada 5 minutos chega uma linha nova na tabela crua

In [6]:
for passo in range(12):
    agora = agora + timedelta(minutes=5)
    print(agora, end=' | ')
    tabela_prev, tabela_rec = rodar_job(bruta[bruta['Data'] <= agora], tabela_prev, tabela_rec)

2026-10-13 07:35:00 | 

linhas cruas lidas: 13 | previsoes novas: 1 | recomendacoes novas: 3
2026-10-13 07:40:00 | 

linhas cruas lidas: 13 | previsoes novas: 1 | recomendacoes novas: 3
2026-10-13 07:45:00 | linhas cruas lidas: 13 | previsoes novas: 1 | recomendacoes novas: 5


2026-10-13 07:50:00 | linhas cruas lidas: 13 | previsoes novas: 1 | recomendacoes novas: 4


2026-10-13 07:55:00 | linhas cruas lidas: 13 | previsoes novas: 1 | recomendacoes novas: 4
2026-10-13 08:00:00 | 

...

linhas cruas lidas: 13 | previsoes novas: 1 | recomendacoes novas: 5
2026-10-13 08:10:00 | 

linhas cruas lidas: 13 | previsoes novas: 1 | recomendacoes novas: 3
2026-10-13 08:15:00 | 

linhas cruas lidas: 13 | previsoes novas: 1 | recomendacoes novas: 5
2026-10-13 08:20:00 | 

linhas cruas lidas: 13 | previsoes novas: 1 | recomendacoes novas: 5
2026-10-13 08:25:00 | 

linhas cruas lidas: 13 | previsoes novas: 1 | recomendacoes novas: 5
2026-10-13 08:30:00 | linhas cruas lidas: 13 | previsoes novas: 1 | recomendacoes novas: 4


In [7]:
#Running again with no new raw row: nothing is duplicated
tabela_prev, tabela_rec = rodar_job(bruta[bruta['Data'] <= agora], tabela_prev, tabela_rec)

linhas cruas lidas: 12 | previsoes novas: 0 | recomendacoes novas: 0


## As duas tabelas de saida

In [8]:
print('Tabela 1 - previsao:', tabela_prev.shape)
tabela_prev.tail(6)

Tabela 1 - previsao: (15074, 19)


,Data,score_camera,prob_15,alarme_15,prob_30,alarme_30,prob_60,alarme_60,prob_90,alarme_90,prob_120,alarme_120,prob_180,alarme_180,prob_240,alarme_240,prob_360,alarme_360,versao_modelo
15068,2026-10-13 08:05:00,0.002,0.002,0,0.002,0,0.002,0,0.075888,0,0.094368,0,0.189290,1,0.117934,1,0.109019,0,1.1.0
15069,2026-10-13 08:10:00,0.002,0.002,0,0.002,0,0.002,0,0.063544,0,0.097664,0,0.115695,1,0.098329,1,0.113985,0,1.1.0
15070,2026-10-13 08:15:00,0.002,0.002,0,0.002,0,0.002,0,0.065846,0,0.107823,0,0.194779,1,0.129673,1,0.200608,1,1.1.0
15071,2026-10-13 08:20:00,0.001,0.001,0,0.001,0,0.001,0,0.065846,0,0.100778,0,0.123042,1,0.106765,1,0.174017,1,1.1.0
15072,2026-10-13 08:25:00,0.001,0.001,0,0.001,0,0.001,0,0.062472,0,0.097664,0,0.101899,1,0.101219,1,0.149773,0,1.1.0
15073,2026-10-13 08:30:00,0.001,0.001,0,0.001,0,0.001,0,0.066561,0,0.104517,0,0.115290,1,0.082894,1,0.126515,0,1.1.0


In [9]:
print('Tabela 2 - recomendacao:', tabela_rec.shape)
tabela_rec.tail(12)

Tabela 2 - recomendacao: (809, 9)


,Data,risco_atual,risco_recomendado,variavel,valor_atual,valor_recomendado,direcao,validada,versao_modelo
797,2026-10-13 08:20:00,0.114,0.082,306FE002-AI001-R,2.410,1.321,baixar,False,1.1.0
798,2026-10-13 08:20:00,0.114,0.082,406-AI-004-2,63.983,76.876,subir,False,1.1.0
799,2026-10-13 08:20:00,0.114,0.082,406-AI-004-4,189.094,203.638,subir,False,1.1.0
800,2026-10-13 08:25:00,0.103,0.072,306-AIT-004-2,56.460,40.016,baixar,False,1.1.0
801,2026-10-13 08:25:00,0.103,0.072,306-AIT-004-5,141.337,111.582,baixar,False,1.1.0
802,2026-10-13 08:25:00,0.103,0.072,406-AI-004-2,73.699,76.922,subir,False,1.1.0
803,2026-10-13 08:25:00,0.103,0.072,406-AI-004-4,203.884,240.244,subir,False,1.1.0
804,2026-10-13 08:25:00,0.103,0.072,406-AIT-001,2.369,2.723,subir,False,1.1.0
805,2026-10-13 08:30:00,0.099,0.081,306-AIT-004-2,58.262,41.818,baixar,False,1.1.0
806,2026-10-13 08:30:00,0.099,0.081,306-AIT-004-5,151.768,111.582,baixar,False,1.1.0


In [10]:
pasta = PATH_REPORTS / 'simulacao_databricks'
pasta.mkdir(exist_ok=True)
tabela_prev.to_csv(pasta / 'pluma_previsao.csv', index=False)
tabela_rec.to_csv(pasta / 'pluma_recomendacao.csv', index=False)
print('Salvas em', pasta)

Salvas em

 /tmp/claude-0/-home-claude/22bcba14-99e8-5e72-856a-2b90ddb97672/scratchpad/v8/reports/simulacao_databricks
